# Clinical Trial Recruitment - Training with TRL

**Theme #2: Long-Horizon Planning & Instruction Following**

This notebook demonstrates training an RL agent on the Clinical Trial Recruitment environment using Hugging Face TRL.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pratimassaravanan/clinical-recruitment-env/blob/main/notebooks/training_trl.ipynb)

## 1. Setup & Installation

In [ ]:
# Install dependencies
!pip install -q transformers trl accelerate torch numpy pandas matplotlib httpx pydantic

In [ ]:
# Clone the environment repository
!git clone https://github.com/pratimassaravanan/clinical-recruitment-env.git
%cd clinical-recruitment-env

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import matplotlib.pyplot as plt

from env import ClinicalRecruitmentEnv
from models import Action, Observation

print("Environment loaded successfully!")

## 2. Environment Overview

The Clinical Trial Recruitment environment simulates a 180-day clinical trial where an agent must:
- Screen patients for eligibility
- Allocate patients to recruitment sites
- Manage budget and timeline constraints
- Handle delayed consequences and regulatory events

In [ ]:
# Initialize environment
env = ClinicalRecruitmentEnv()
result = env.reset(task="easy_bench")

print(f"Observation fields: {len(result.observation.model_dump())}")
print(f"\nKey observation values:")
print(f"  - Timestamp: {result.observation.timestamp}")
print(f"  - Budget: ${result.observation.budget_remaining:,.0f}")
print(f"  - Target enrollment: {result.observation.target_enrollment}")
print(f"  - Available patients: {len(result.observation.available_patients)}")
print(f"  - Sites: {len(result.observation.site_performance)}")

## 3. Define Policy Network

In [ ]:
# Action space
ACTION_SPACE = [
    "screen_patient",
    "recontact",
    "allocate_to_site",
    "adjust_strategy",
    "stop_recruitment",
    "request_budget_extension",
    "negotiate_site_terms",
    "plan_next_phase",
    "summarize_and_index",
    "retrieve_relevant_history",
]

STATE_DIM = 37
ACTION_DIM = len(ACTION_SPACE)

def extract_features(obs: dict) -> np.ndarray:
    """Extract 37-dimensional feature vector from observation."""
    funnel = obs.get("current_funnel", {})
    milestones = obs.get("milestones", {})
    uncertainty = obs.get("uncertainty_components", {})
    
    features = [
        # Core state (12)
        obs.get("timestamp", 0) / 180.0,
        obs.get("budget_remaining", 0) / 150000.0,
        obs.get("time_to_deadline_days", 180) / 180.0,
        obs.get("enrolled_so_far", 0) / 150.0,
        obs.get("target_enrollment", 100) / 150.0,
        obs.get("uncertainty_level", 0.5),
        obs.get("difficulty", 1) / 3.0,
        obs.get("dropout_rate_7d", 0.0),
        obs.get("screening_backlog", 0) / 20.0,
        obs.get("milestone_potential", 0.0),
        min(1.0, obs.get("token_budget_remaining", 12000) / 12000.0),
        obs.get("token_efficiency_score", 1.0),
        
        # Funnel state (6)
        funnel.get("contacted", 0) / 100.0,
        funnel.get("screened", 0) / 100.0,
        funnel.get("eligible", 0) / 100.0,
        funnel.get("consented", 0) / 100.0,
        funnel.get("enrolled", 0) / 100.0,
        funnel.get("dropped", 0) / 50.0,
        
        # Milestones (4)
        float(milestones.get("25%", False)),
        float(milestones.get("50%", False)),
        float(milestones.get("75%", False)),
        float(milestones.get("100%", False)),
        
        # Uncertainty decomposition (3)
        uncertainty.get("patient_pool", 0.3),
        uncertainty.get("site_operations", 0.3),
        uncertainty.get("policy_dynamics", 0.3),
        
        # Long-horizon signals (8)
        obs.get("delayed_effects_pending", 0) / 10.0,
        len(obs.get("available_patients", [])) / 5.0,
        len(obs.get("site_performance", {})) / 5.0,
        len(obs.get("recent_events", [])) / 5.0,
        float(obs.get("hindsight_available", False)),
        len(obs.get("indexed_memory_summary", {})) / 10.0,
        len(obs.get("current_plan", {})) / 5.0,
        obs.get("hypothesis_accuracy", 0.0),
        
        # Derived features (4)
        obs.get("enrolled_so_far", 0) / max(1, obs.get("target_enrollment", 100)),
        (obs.get("target_enrollment", 100) - obs.get("enrolled_so_far", 0)) / max(1, obs.get("time_to_deadline_days", 180)),
        funnel.get("consented", 0) / max(1, funnel.get("eligible", 1)),
        funnel.get("enrolled", 0) / max(1, funnel.get("consented", 1)),
    ]
    
    return np.array(features, dtype=np.float32)

print(f"State dimension: {STATE_DIM}")
print(f"Action dimension: {ACTION_DIM}")

In [ ]:
class ActorCritic(nn.Module):
    """Actor-Critic network for PPO training."""
    
    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = 128):
        super().__init__()
        
        # Shared backbone
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        
        # Actor head (policy)
        self.actor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, action_dim),
        )
        
        # Critic head (value)
        self.critic = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
        )
    
    def forward(self, state):
        shared = self.shared(state)
        action_logits = self.actor(shared)
        value = self.critic(shared)
        return action_logits, value
    
    def get_action(self, state, deterministic=False):
        action_logits, value = self.forward(state)
        probs = torch.softmax(action_logits, dim=-1)
        
        if deterministic:
            action = torch.argmax(probs, dim=-1)
        else:
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()
        
        log_prob = torch.log(probs.gather(-1, action.unsqueeze(-1)).squeeze(-1) + 1e-8)
        return action, log_prob, value.squeeze(-1)

# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ActorCritic(STATE_DIM, ACTION_DIM).to(device)
optimizer = optim.Adam(model.parameters(), lr=3e-4)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {device}")

## 4. PPO Training Loop

In [ ]:
def build_action(action_idx: int, obs: dict) -> dict:
    """Build action dict from action index."""
    action_type = ACTION_SPACE[action_idx]
    action_dict = {"action_type": action_type}
    
    # Add required parameters
    patients = obs.get("available_patients", [])
    sites = obs.get("site_performance", {})
    
    if action_type in ["screen_patient", "recontact", "allocate_to_site"]:
        if patients:
            action_dict["patient_id"] = patients[0].get("id")
    
    if action_type == "allocate_to_site" and sites:
        action_dict["site_id"] = list(sites.keys())[0]
    
    if action_type == "adjust_strategy":
        action_dict["strategy_change"] = "increase_outreach"
    
    if action_type == "plan_next_phase":
        action_dict["target_phase"] = "screening"
    
    return action_dict


def collect_trajectory(env, model, task_id="easy_bench", max_steps=180):
    """Collect a single trajectory."""
    result = env.reset(task=task_id)
    obs_dict = result.observation.model_dump()
    
    states, actions, rewards, log_probs, values, dones = [], [], [], [], [], []
    
    for step in range(max_steps):
        if result.done:
            break
        
        # Get state
        state = torch.FloatTensor(extract_features(obs_dict)).unsqueeze(0).to(device)
        
        # Get action
        with torch.no_grad():
            action, log_prob, value = model.get_action(state)
        
        action_idx = action.item()
        action_dict = build_action(action_idx, obs_dict)
        
        # Step environment
        result = env.step(Action(**action_dict))
        
        # Store transition
        states.append(state.squeeze(0))
        actions.append(action_idx)
        rewards.append(result.reward)
        log_probs.append(log_prob.item())
        values.append(value.item())
        dones.append(result.done)
        
        obs_dict = result.observation.model_dump()
    
    return {
        "states": torch.stack(states),
        "actions": torch.tensor(actions, dtype=torch.long),
        "rewards": rewards,
        "log_probs": torch.tensor(log_probs),
        "values": torch.tensor(values),
        "dones": dones,
        "final_score": result.info.get("final_score", 0.0),
    }

In [ ]:
def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    """Compute Generalized Advantage Estimation."""
    advantages = []
    gae = 0
    
    for t in reversed(range(len(rewards))):
        if t == len(rewards) - 1:
            next_value = 0
        else:
            next_value = values[t + 1]
        
        delta = rewards[t] + gamma * next_value * (1 - dones[t]) - values[t]
        gae = delta + gamma * lam * (1 - dones[t]) * gae
        advantages.insert(0, gae)
    
    return torch.tensor(advantages, dtype=torch.float32)


def ppo_update(model, optimizer, states, actions, old_log_probs, advantages, returns,
               clip_epsilon=0.2, epochs=4, batch_size=64):
    """PPO policy update."""
    states = states.to(device)
    actions = actions.to(device)
    old_log_probs = old_log_probs.to(device)
    advantages = advantages.to(device)
    returns = returns.to(device)
    
    # Normalize advantages
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    
    total_loss = 0
    n_updates = 0
    
    for _ in range(epochs):
        indices = torch.randperm(len(states))
        
        for start in range(0, len(states), batch_size):
            end = start + batch_size
            batch_idx = indices[start:end]
            
            batch_states = states[batch_idx]
            batch_actions = actions[batch_idx]
            batch_old_log_probs = old_log_probs[batch_idx]
            batch_advantages = advantages[batch_idx]
            batch_returns = returns[batch_idx]
            
            # Forward pass
            action_logits, values = model(batch_states)
            probs = torch.softmax(action_logits, dim=-1)
            new_log_probs = torch.log(probs.gather(-1, batch_actions.unsqueeze(-1)).squeeze(-1) + 1e-8)
            
            # PPO loss
            ratio = torch.exp(new_log_probs - batch_old_log_probs)
            surr1 = ratio * batch_advantages
            surr2 = torch.clamp(ratio, 1 - clip_epsilon, 1 + clip_epsilon) * batch_advantages
            policy_loss = -torch.min(surr1, surr2).mean()
            
            # Value loss
            value_loss = 0.5 * ((values.squeeze(-1) - batch_returns) ** 2).mean()
            
            # Entropy bonus
            entropy = -(probs * torch.log(probs + 1e-8)).sum(-1).mean()
            
            # Total loss
            loss = policy_loss + 0.5 * value_loss - 0.01 * entropy
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
            
            total_loss += loss.item()
            n_updates += 1
    
    return total_loss / max(1, n_updates)

## 5. Training

In [ ]:
# Training parameters
N_EPISODES = 100
GAMMA = 0.99
LAM = 0.95

# Track metrics
episode_rewards = []
episode_scores = []
losses = []

print("Starting PPO training...")
print("=" * 50)

for episode in range(N_EPISODES):
    # Collect trajectory
    task = ["easy_bench", "medium_bench", "hard_bench"][episode % 3]
    traj = collect_trajectory(env, model, task_id=task)
    
    # Compute advantages and returns
    advantages = compute_gae(traj["rewards"], traj["values"].tolist(), traj["dones"], GAMMA, LAM)
    returns = advantages + traj["values"]
    
    # PPO update
    loss = ppo_update(
        model, optimizer,
        traj["states"], traj["actions"], traj["log_probs"],
        advantages, returns
    )
    
    # Track metrics
    total_reward = sum(traj["rewards"])
    episode_rewards.append(total_reward)
    episode_scores.append(traj["final_score"])
    losses.append(loss)
    
    # Log progress
    if (episode + 1) % 10 == 0:
        avg_reward = np.mean(episode_rewards[-10:])
        avg_score = np.mean(episode_scores[-10:])
        print(f"Episode {episode + 1}/{N_EPISODES} | "
              f"Avg Reward: {avg_reward:.3f} | "
              f"Avg Score: {avg_score:.4f} | "
              f"Loss: {loss:.4f}")

print("\nTraining complete!")

## 6. Visualize Training Progress

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Rewards
ax = axes[0]
ax.plot(episode_rewards, alpha=0.3, color='blue')
ax.plot(np.convolve(episode_rewards, np.ones(10)/10, mode='valid'), color='blue', linewidth=2)
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.set_title('Episode Rewards')
ax.grid(alpha=0.3)

# Scores
ax = axes[1]
ax.plot(episode_scores, alpha=0.3, color='green')
ax.plot(np.convolve(episode_scores, np.ones(10)/10, mode='valid'), color='green', linewidth=2)
ax.set_xlabel('Episode')
ax.set_ylabel('Final Score')
ax.set_title('Final Scores')
ax.grid(alpha=0.3)

# Loss
ax = axes[2]
ax.plot(losses, color='red')
ax.set_xlabel('Episode')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

print(f"\nFinal metrics:")
print(f"  Average reward (last 10): {np.mean(episode_rewards[-10:]):.3f}")
print(f"  Average score (last 10): {np.mean(episode_scores[-10:]):.4f}")
print(f"  Best score: {max(episode_scores):.4f}")

## 7. Evaluate Trained Agent

In [ ]:
# Evaluate on all tasks
model.eval()
eval_results = {}

for task in ["easy_bench", "medium_bench", "hard_bench"]:
    scores = []
    for _ in range(5):
        with torch.no_grad():
            traj = collect_trajectory(env, model, task_id=task)
        scores.append(traj["final_score"])
    
    eval_results[task] = {
        "mean": np.mean(scores),
        "std": np.std(scores),
        "scores": scores
    }
    print(f"{task}: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")

print(f"\nOverall: {np.mean([r['mean'] for r in eval_results.values()]):.4f}")

## 8. Before/After Comparison

In [ ]:
# Compare with random baseline
def random_baseline(env, task_id, n_episodes=5):
    scores = []
    for _ in range(n_episodes):
        result = env.reset(task=task_id)
        obs = result.observation.model_dump()
        
        while not result.done:
            action_idx = np.random.randint(0, ACTION_DIM)
            action_dict = build_action(action_idx, obs)
            result = env.step(Action(**action_dict))
            obs = result.observation.model_dump()
        
        scores.append(result.info.get("final_score", 0.0))
    return np.mean(scores)

# Comparison
print("Before/After Comparison")
print("=" * 40)
print(f"{'Task':<15} {'Random':<12} {'Trained':<12} {'Improvement':<12}")
print("-" * 40)

for task in ["easy_bench", "medium_bench", "hard_bench"]:
    random_score = random_baseline(env, task)
    trained_score = eval_results[task]["mean"]
    improvement = (trained_score - random_score) / max(random_score, 0.01) * 100
    print(f"{task:<15} {random_score:<12.4f} {trained_score:<12.4f} {improvement:+.1f}%")

## 9. Save Model

In [ ]:
# Save trained model
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'episode_rewards': episode_rewards,
    'episode_scores': episode_scores,
    'eval_results': eval_results,
}, 'trained_model.pt')

print("Model saved to trained_model.pt")

## Summary

This notebook demonstrated:

1. **Environment Setup**: Loading the Clinical Trial Recruitment environment
2. **Policy Network**: Actor-Critic architecture with 37-dim state input
3. **PPO Training**: 100 episodes with GAE and clipped objective
4. **Evaluation**: Testing on all difficulty levels
5. **Improvement**: Clear before/after comparison showing training progress

### Key Results

- The trained agent shows improvement over random baseline
- Long-horizon signals (milestones, delayed effects) help guide learning
- Token efficiency rewards encourage efficient decision-making

### Next Steps

- Try the advanced agents (HCAPO, MiRA, KLong, MemexRL)
- Increase training episodes for better convergence
- Experiment with curriculum learning